In [ ]:
import sqlite3
import pandas as pd

conexao = sqlite3.connect("loja_pai.db")
cursor = conexao.cursor()

df = pd.read_csv('dados_sujos.csv')

In [ ]:

df['Nome_Cliente'] = (df['Nome_Cliente'].str.strip().str.title())

In [ ]:
# O 'mixed' avisa o Pandas que a coluna tem mais de um formato de data
df['Data'] = pd.to_datetime(
    df['Data'], format='mixed', dayfirst=True, errors='coerce'
)

In [ ]:
# emove tudo que não for númro ou vírgula, troca a vírgula por ponto e converte
df['Valor'] = (
    df['Valor']
    .astype(str)
    .str.replace(r'[^\d,]', '', regex=True)
    .str.replace(',', '.')
    .astype(float)
)

In [ ]:
df['Pago'] = df['Pago'].str.title().str.strip()
df['Pago'] = df['Pago'].map({'Sim':'1', 'Não':'0', '0':'0', '1':'1'})

In [ ]:
df['Pago'] = df['Pago'].astype(int)

In [ ]:
df.columns = df.columns.str.lower()
df = df.rename(columns={'nome_cliente':'nome'})
df =df.rename(columns={'tipo_servico':'servico'})

In [ ]:
df["valor"] = df["valor"].fillna(0)


In [ ]:
novos_clientes = df['nome'].unique()
velhos_clientes_df = pd.read_sql_query('SELECT nome FROM Cliente', conexao)
#set transforma a coluna em array
velhos_clientes_set = set(velhos_clientes_df['nome'])
print(velhos_clientes_set)
for cliente in novos_clientes:
    if cliente not in velhos_clientes_set:
        cursor.execute('INSERT INTO Cliente(nome, cpf, telefone) VALUES(?, ?, ?)', (cliente, '0000', '0000'))

conexao.commit()

In [ ]:
for i in df.index:
    id = cursor.execute('SELECT id_cliente FROM Cliente WHERE  nome = ?', (df.loc[i, 'nome'],)).fetchone()[0]
    cursor.execute("""
            INSERT INTO Servico (id_cliente, id_funcionario, aparelho, servico, valor, endereco, hora, observacao, data, status, pago)
            VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            id, 
            1, 
            df.loc[i, 'aparelho'], 
            df.loc[i, 'servico'], 
            df.loc[i, 'valor'], 
            df.loc[i, 'endereco'], '', '', 
            df.loc[i, 'data'].strftime('%Y-%m-%d'), 
            'pendente', 
            int(df.loc[i, 'pago'])
        ))

conexao.commit()


In [ ]:
cursor.execute("UPDATE Servico SET servico = 'Instalacao' WHERE servico = 'Instalação';")
cursor.execute("UPDATE Servico SET servico = 'Manutencao' WHERE servico = 'Manutenção';")
cursor.execute("UPDATE Servico SET servico = 'Visita' WHERE servico = 'Visita Técnica';")
conexao.commit()

In [ ]:
print(pd.read_sql_query("""
SELECT
    servico AS tipo_servico,
    ROUND(AVG(valor),2) AS media_cobrada
FROM Servico 
GROUP BY servico;
""", conexao))

In [ ]:
print(pd.read_sql_query("""
SELECT 
    aparelho,
    COUNT(status) AS pendentes
FROM Servico
WHERE status = 'pendente'
GROUP BY aparelho;

""", conexao))

In [ ]:
print(pd.read_sql_query("""
SELECT
    c.nome,
    SUM(s.valor)AS gasto
FROM Cliente c
JOIN Servico s ON c.id_cliente = s.id_cliente
GROUP BY nome
ORDER BY gasto DESC
LIMIT 5;
""", conexao))